# High-Performance Dataset Cleaning v3

Notebook version optimized for a local workstation with large CSV/TXT/Parquet files.

Key changes:
- Streams large files in chunks instead of loading full dataframes.
- Writes cleaned data as chunked Parquet for faster downstream training.
- Uses Polars automatically for normal UTF-8 CSV files when available.
- Uses robust pandas chunking for huge/headerless LANL text files.
- Supports single-thread and multi-process execution.
- Auto-detects CPU/RAM and logs long-running progress.


In [1]:
# Run this cell once if imports fail, then restart the kernel and run cells again.
import sys
print('Python executable:', sys.executable)
%pip install -U pandas pyarrow psutil polars


Python executable: e:\thesis\.venv\Scripts\python.exe
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import argparse

# ---- Notebook configuration -------------------------------------------------
INPUT_FOLDER = Path(r'E:/thesis/dataset')
OUTPUT_FOLDER = INPUT_FOLDER / 'dataset_cleaned_parquet'

# 0 = auto-detect from CPU/RAM. Ryzen 5 5500 + 32GB RAM usually works well with 4-6.
NUM_WORKERS = 4

# Rows per chunk. Increase to 750_000 or 1_000_000 if RAM stays comfortable.
CHUNK_SIZE = 500_000

# 'multi-process' or 'single-thread'
MODE = 'multi-process'

# 'auto', 'pandas', or 'polars'. Auto uses Polars for safe CSVs and pandas for huge/headerless TXT.
ENGINE = 'auto'

COMPRESSION = 'snappy'
OVERWRITE = False
SAMPLE_ROWS = 4096
MAX_FILES = 0  # use a small number like 1 for smoke tests
LOG_LEVEL = 'INFO'
INCLUDE_EXISTING_CLEANED = False

print('INPUT_FOLDER =', INPUT_FOLDER)
print('OUTPUT_FOLDER =', OUTPUT_FOLDER)


INPUT_FOLDER = E:\thesis\dataset
OUTPUT_FOLDER = E:\thesis\dataset\dataset_cleaned_parquet


In [ ]:
# ---- Optimized cleaning implementation --------------------------------------
# This cell is intentionally self-contained so the notebook can run without
# depending on the separate .py entry point.
"""
High-performance local dataset cleaner.

Designed for large CSV/TXT datasets on a workstation:
- streams input files in chunks instead of loading whole files,
- processes independent files in parallel,
- writes chunked Parquet output for fast downstream training,
- uses optional Polars for CSV files when available,
- keeps pandas + pyarrow as the reliable default for very large/headerless files.
"""

from __future__ import annotations

import argparse
import csv
import gc
import json
import logging
import math
import os
import re
import shutil
import sys
import time
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Iterable



try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except Exception as exc:  # pragma: no cover - fail fast for production use
    raise RuntimeError(
        "pyarrow is required. Install with: pip install pyarrow"
    ) from exc

try:
    import psutil
except Exception:  # pragma: no cover - optional
    psutil = None

try:
    import polars as pl
except Exception:  # pragma: no cover - optional
    pl = None


def ensure_runtime_dependencies() -> None:
    """Rebind notebook globals after kernel restarts or partial cell execution."""
    global pd, pa, pq, psutil, pl
    import pandas as pd
    import pyarrow as pa
    import pyarrow.parquet as pq
    try:
        import psutil
    except Exception:
        psutil = None
    try:
        import polars as pl
    except Exception:
        pl = None


DELIMITER_CANDIDATES = (",", "\t", "|", ";")
ENCODING_FALLBACKS = ("utf-8", "utf-8-sig", "latin-1", "cp1252")
INPUT_SUFFIXES = {".csv", ".txt", ".tsv", ".parquet"}
DEFAULT_EXCLUDE_DIRS = {
    ".git",
    ".ipynb_checkpoints",
    "__pycache__",
    "dataset_cleaned_parquet",
    "dataset_cleaned",
}

LANL_SCHEMAS = {
    "auth": [
        "time",
        "source_user",
        "destination_user",
        "source_computer",
        "destination_computer",
        "authentication_type",
        "logon_type",
        "authentication_orientation",
        "success_failure",
    ],
    "proc": ["time", "user", "computer", "process", "event_type"],
    "flows": [
        "time",
        "duration",
        "source_computer",
        "source_port",
        "destination_computer",
        "destination_port",
        "protocol",
        "packet_count",
        "byte_count",
    ],
    "dns": ["time", "source_computer", "resolved_computer"],
    "redteam": ["time", "user", "source_computer", "destination_computer"],
}


@dataclass(frozen=True)
class FilePlan:
    path: str
    rel_parent: str
    stem_slug: str
    suffix: str
    size_bytes: int


@dataclass
class FileResult:
    file: str
    output: str | None
    rows: int
    parts: int
    input_mb: float
    output_mb: float
    seconds: float
    engine: str
    error: str | None = None


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Stream-clean large CSV/TXT/Parquet datasets into chunked Parquet."
    )
    parser.add_argument("--input-folder", default="dataset", help="Raw dataset folder.")
    parser.add_argument(
        "--output-folder",
        default=str(Path("dataset") / "dataset_cleaned_parquet"),
        help="Folder for cleaned Parquet datasets.",
    )
    parser.add_argument(
        "--num-workers",
        type=int,
        default=0,
        help="Parallel file workers. 0 auto-detects from CPU/RAM.",
    )
    parser.add_argument(
        "--chunk-size",
        type=int,
        default=500_000,
        help="Rows per pandas chunk for CSV/TXT streaming.",
    )
    parser.add_argument(
        "--mode",
        choices=("single-thread", "multi-process"),
        default="multi-process",
        help="Execution mode.",
    )
    parser.add_argument(
        "--engine",
        choices=("auto", "pandas", "polars"),
        default="auto",
        help="Processing engine. Polars is used only when it is safe for the file.",
    )
    parser.add_argument(
        "--compression",
        default="snappy",
        choices=("snappy", "zstd", "gzip", "brotli", "none"),
        help="Parquet compression.",
    )
    parser.add_argument(
        "--overwrite",
        action="store_true",
        help="Replace existing output folder.",
    )
    parser.add_argument(
        "--sample-rows",
        type=int,
        default=4096,
        help="Rows used for dtype inference per file.",
    )
    parser.add_argument(
        "--include-existing-cleaned",
        action="store_true",
        help="Also scan folders that look like previous cleaned outputs.",
    )
    parser.add_argument(
        "--max-files",
        type=int,
        default=0,
        help="Optional smoke-test limit. 0 processes all files.",
    )
    parser.add_argument(
        "--log-level",
        default="INFO",
        choices=("DEBUG", "INFO", "WARNING", "ERROR"),
        help="Console logging level.",
    )
    return parser.parse_args()


def configure_logging(level: str) -> None:
    logging.basicConfig(
        level=getattr(logging, level),
        format="%(asctime)s | %(levelname)-7s | %(processName)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )


def normalize_column(name: object) -> str:
    value = str(name).strip().lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value or "unnamed_column"


def make_unique_columns(columns: Iterable[object]) -> list[str]:
    seen: dict[str, int] = {}
    output: list[str] = []
    for column in columns:
        base = normalize_column(column)
        count = seen.get(base, 0)
        seen[base] = count + 1
        output.append(base if count == 0 else f"{base}_{count}")
    return output


def slugify(value: str) -> str:
    slug = re.sub(r"[^a-zA-Z0-9]+", "_", value).strip("_").lower()
    return slug or "dataset"


def detect_delimiter(path: Path, encoding: str) -> str:
    with path.open("r", encoding=encoding, errors="replace", newline="") as handle:
        sample = handle.read(256 * 1024)
    try:
        dialect = csv.Sniffer().sniff(sample, delimiters="".join(DELIMITER_CANDIDATES))
        return dialect.delimiter
    except Exception:
        counts = {delimiter: sample.count(delimiter) for delimiter in DELIMITER_CANDIDATES}
        best = max(counts, key=counts.get)
        return best if counts[best] > 0 else ","


def first_nonempty_line(path: Path, encoding: str) -> str:
    with path.open("r", encoding=encoding, errors="replace", newline="") as handle:
        for line in handle:
            stripped = line.strip()
            if stripped:
                return stripped
    return ""


def looks_like_header(fields: list[str]) -> bool:
    if not fields:
        return False
    alphaish = sum(bool(re.search(r"[A-Za-z_]", field)) for field in fields)
    numericish = sum(_is_number(field) for field in fields)
    return alphaish >= max(1, math.ceil(len(fields) * 0.5)) and numericish == 0


def _is_number(value: str) -> bool:
    try:
        float(value)
        return True
    except Exception:
        return False


def lanl_schema_for(path: Path) -> list[str] | None:
    names = [path.stem.lower(), path.parent.name.lower()]
    for name in names:
        key = name.replace(".txt", "")
        if key in LANL_SCHEMAS:
            return LANL_SCHEMAS[key]
    return None


def read_csv_metadata(path: Path, sample_rows: int) -> dict:
    errors: list[str] = []
    for encoding in ENCODING_FALLBACKS:
        try:
            delimiter = detect_delimiter(path, encoding)
            first_line = first_nonempty_line(path, encoding)
            fields = next(csv.reader([first_line], delimiter=delimiter)) if first_line else []
            lanl_columns = lanl_schema_for(path)
            has_header = looks_like_header(fields) and lanl_columns is None

            if has_header:
                sample = pd.read_csv(
                    path,
                    sep=delimiter,
                    encoding=encoding,
                    nrows=sample_rows,
                    low_memory=False,
                    on_bad_lines="skip",
                )
            else:
                names = lanl_columns or [f"column_{idx + 1}" for idx in range(len(fields))]
                sample = pd.read_csv(
                    path,
                    sep=delimiter,
                    encoding=encoding,
                    names=names,
                    header=None,
                    nrows=sample_rows,
                    low_memory=False,
                    on_bad_lines="skip",
                )

            sample.columns = make_unique_columns(sample.columns)
            dtype_map = infer_read_dtypes(sample)
            return {
                "encoding": encoding,
                "delimiter": delimiter,
                "has_header": has_header,
                "names": None if has_header else list(sample.columns),
                "columns": list(sample.columns),
                "dtype_map": dtype_map,
            }
        except Exception as exc:
            errors.append(f"{encoding}: {exc}")
    raise RuntimeError("Could not parse file metadata: " + " | ".join(errors[:4]))


def infer_read_dtypes(sample: pd.DataFrame) -> dict[str, str]:
    dtype_map: dict[str, str] = {}
    for column in sample.columns:
        series = sample[column]
        if pd.api.types.is_bool_dtype(series):
            dtype_map[column] = "boolean"
            continue
        if pd.api.types.is_integer_dtype(series):
            dtype_map[column] = "float64"
            continue
        if pd.api.types.is_float_dtype(series):
            dtype_map[column] = "float64"
            continue

        coerced = pd.to_numeric(series, errors="coerce")
        non_null = max(int(series.notna().sum()), 1)
        numeric_ratio = float(coerced.notna().sum()) / non_null
        dtype_map[column] = "float64" if numeric_ratio >= 0.98 else "string"
    return dtype_map


def clean_frame(df: pd.DataFrame, source_file: str, dtype_map: dict[str, str]) -> pd.DataFrame:
    df = df.copy(deep=False)
    df.columns = make_unique_columns(df.columns)
    df = df.loc[:, ~pd.Index(df.columns).duplicated()]

    for column, dtype in dtype_map.items():
        if column not in df.columns:
            continue
        if dtype == "string":
            series = df[column].astype("string")
            series = series.str.strip()
            df[column] = series.mask(series.isin(["", "?", "nan", "None", "NULL", "null"]))
        else:
            df[column] = pd.to_numeric(df[column], errors="coerce", downcast="float")

    unseen_columns = [column for column in df.columns if column not in dtype_map]
    for column in unseen_columns:
        if column == "source_file":
            continue
        df[column] = df[column].astype("string").str.strip()

    df["source_file"] = source_file
    return df


def dataframe_to_table(df: pd.DataFrame) -> pa.Table:
    return pa.Table.from_pandas(df, preserve_index=False)


def write_table_part(table: pa.Table, output_dir: Path, part_idx: int, compression: str) -> Path:
    compression_arg = None if compression == "none" else compression
    part_path = output_dir / f"part-{part_idx:05d}.parquet"
    pq.write_table(
        table,
        part_path,
        compression=compression_arg,
        use_dictionary=True,
        write_statistics=True,
    )
    return part_path


def make_staging_dir(final_dir: Path) -> Path:
    suffix = f"__staging_{os.getpid()}_{time.time_ns()}"
    return final_dir.with_name(final_dir.name + suffix)


def commit_staging_dir(staging_dir: Path, final_dir: Path) -> None:
    """Replace final_dir only after staging_dir has been fully written."""
    backup_dir = final_dir.with_name(f"{final_dir.name}__backup_{os.getpid()}_{time.time_ns()}")
    moved_existing = False
    try:
        if final_dir.exists():
            final_dir.rename(backup_dir)
            moved_existing = True
        staging_dir.rename(final_dir)
    except Exception:
        if moved_existing and backup_dir.exists() and not final_dir.exists():
            backup_dir.rename(final_dir)
        raise
    else:
        if backup_dir.exists():
            shutil.rmtree(backup_dir, ignore_errors=True)


def discard_staging_dir(staging_dir: Path) -> None:
    if staging_dir.exists():
        shutil.rmtree(staging_dir, ignore_errors=True)


def process_csv_like_file(plan: FilePlan, output_dir: Path, args: argparse.Namespace) -> FileResult:
    path = Path(plan.path)
    start = time.perf_counter()
    file_output_dir = output_dir / plan.rel_parent / plan.stem_slug
    tmp_output_dir = make_staging_dir(file_output_dir)
    tmp_output_dir.mkdir(parents=True, exist_ok=True)

    metadata = read_csv_metadata(path, args.sample_rows)
    if should_use_polars(path, metadata, args):
        return process_csv_polars_file(plan, output_dir, args, metadata, start, file_output_dir, tmp_output_dir)

    read_kwargs = {
        "sep": metadata["delimiter"],
        "encoding": metadata["encoding"],
        "chunksize": args.chunk_size,
        "low_memory": False,
        "on_bad_lines": "skip",
        "dtype": metadata["dtype_map"],
    }
    if metadata["has_header"]:
        read_kwargs["header"] = 0
    else:
        read_kwargs["header"] = None
        read_kwargs["names"] = metadata["names"]

    rows = 0
    parts = 0
    reader = pd.read_csv(path, **read_kwargs)
    for parts, chunk in enumerate(reader, start=1):
        cleaned = clean_frame(chunk, path.name, metadata["dtype_map"])
        rows += len(cleaned)
        write_table_part(dataframe_to_table(cleaned), tmp_output_dir, parts, args.compression)
        if parts == 1 or parts % 10 == 0:
            logging.info(
                "%s | wrote %s chunks, %s rows",
                path.name,
                f"{parts:,}",
                f"{rows:,}",
            )
        del chunk, cleaned
        gc.collect()

    commit_staging_dir(tmp_output_dir, file_output_dir)

    elapsed = time.perf_counter() - start
    output_bytes = folder_size(file_output_dir)
    return FileResult(
        file=str(path),
        output=str(file_output_dir),
        rows=rows,
        parts=parts,
        input_mb=plan.size_bytes / 1_000_000,
        output_mb=output_bytes / 1_000_000,
        seconds=elapsed,
        engine="pandas",
    )


def should_use_polars(path: Path, metadata: dict, args: argparse.Namespace) -> bool:
    if args.engine == "pandas":
        return False
    if pl is None:
        return False
    if path.suffix.lower() != ".csv":
        return False
    if metadata["encoding"] not in {"utf-8", "utf-8-sig"}:
        return False
    if not metadata["has_header"]:
        return False
    return args.engine in {"auto", "polars"}


def process_csv_polars_file(
    plan: FilePlan,
    output_dir: Path,
    args: argparse.Namespace,
    metadata: dict,
    start: float, 
    file_output_dir: Path,
    tmp_output_dir: Path,
) -> FileResult:
    path = Path(plan.path)

    tmp_output_dir.mkdir(parents=True, exist_ok=True)

    compression_arg = "uncompressed" if args.compression == "none" else args.compression
    lazy_frame = pl.scan_csv(
        str(path),
        separator=metadata["delimiter"],
        has_header=True,
        with_column_names=make_unique_columns,
        infer_schema_length=args.sample_rows,
        ignore_errors=True,
        truncate_ragged_lines=True,
        null_values=["", "?", "nan", "None", "NULL", "null"],
    )

    rows = 0
    parts = 0
    batch_iter = lazy_frame.collect_batches(chunk_size=args.chunk_size)
    for batch in batch_iter:
        parts += 1
        cleaned = clean_polars_frame(batch, path.name)
        rows += cleaned.height
        part_path = tmp_output_dir / f"part-{parts:05d}.parquet"
        cleaned.write_parquet(
            part_path,
            compression=compression_arg,
            statistics=True,
        )
        if parts == 1 or parts % 10 == 0:
            logging.info(
                "%s | wrote %s chunks, %s rows",
                path.name,
                f"{parts:,}",
                f"{rows:,}",
            )

    commit_staging_dir(tmp_output_dir, file_output_dir)

    elapsed = time.perf_counter() - start
    output_bytes = folder_size(file_output_dir)
    return FileResult(
        file=str(path),
        output=str(file_output_dir),
        rows=rows,
        parts=parts,
        input_mb=plan.size_bytes / 1_000_000,
        output_mb=output_bytes / 1_000_000,
        seconds=elapsed,
        engine="polars",
    )


def clean_polars_frame(df: "pl.DataFrame", source_file: str) -> "pl.DataFrame":
    rename_map = dict(zip(df.columns, make_unique_columns(df.columns)))
    df = df.rename(rename_map)

    string_columns = [column for column, dtype in zip(df.columns, df.dtypes) if dtype == pl.String]
    if string_columns:
        df = df.with_columns(
            [
                pl.col(column)
                .str.strip_chars()
                .replace(["", "?", "nan", "None", "NULL", "null"], None)
                .alias(column)
                for column in string_columns
            ]
        )
    return df.with_columns(pl.lit(source_file).alias("source_file"))


def process_parquet_file(plan: FilePlan, output_dir: Path, args: argparse.Namespace) -> FileResult:
    path = Path(plan.path)
    start = time.perf_counter()
    file_output_dir = output_dir / plan.rel_parent / plan.stem_slug
    tmp_output_dir = make_staging_dir(file_output_dir)
    tmp_output_dir.mkdir(parents=True, exist_ok=True)

    parquet_file = pq.ParquetFile(path)
    rows = 0
    parts = 0
    logging.info("%s | start streaming parquet batches", path.name)
    for parts, batch in enumerate(parquet_file.iter_batches(batch_size=args.chunk_size), start=1):
        table = pa.Table.from_batches([batch])
        table = table.rename_columns(make_unique_columns(table.column_names))
        source = pa.array([path.name] * table.num_rows, type=pa.large_string())
        table = table.append_column("source_file", source)
        rows += table.num_rows
        write_table_part(table, tmp_output_dir, parts, args.compression)
        if parts == 1 or parts % 10 == 0:
            logging.info("%s | wrote %s parquet batches, %s rows", path.name, f"{parts:,}", f"{rows:,}")
        del table, batch
        gc.collect()

    commit_staging_dir(tmp_output_dir, file_output_dir)

    elapsed = time.perf_counter() - start
    output_bytes = folder_size(file_output_dir)
    return FileResult(
        file=str(path),
        output=str(file_output_dir),
        rows=rows,
        parts=parts,
        input_mb=plan.size_bytes / 1_000_000,
        output_mb=output_bytes / 1_000_000,
        seconds=elapsed,
        engine="pyarrow",
    )


def process_file(plan: FilePlan, output_folder: str, args_dict: dict) -> FileResult:
    ensure_runtime_dependencies()
    args = argparse.Namespace(**args_dict)
    output_dir = Path(output_folder)
    try:
        logging.info("Start processing %s", plan.path)
        suffix = plan.suffix.lower()
        if suffix == ".parquet":
            result = process_parquet_file(plan, output_dir, args)
        else:
            result = process_csv_like_file(plan, output_dir, args)
        logging.info("Successfully processed %s", plan.path)
        return result
    except Exception as exc:
        logging.exception("Failed processing %s", plan.path)
        final_dir = output_dir / plan.rel_parent / plan.stem_slug
        for staging_dir in final_dir.parent.glob(final_dir.name + "__staging_*"):
            discard_staging_dir(staging_dir)
        return FileResult(
            file=plan.path,
            output=None,
            rows=0,
            parts=0,
            input_mb=plan.size_bytes / 1_000_000,
            output_mb=0.0,
            seconds=0.0,
            engine=args.engine,
            error=f"{type(exc).__name__}: {exc}",
        )


def should_skip_path(path: Path, output_dir: Path, include_existing_cleaned: bool) -> bool:
    try:
        path.relative_to(output_dir)
        return True
    except ValueError:
        pass

    if include_existing_cleaned:
        return False
    return any(part in DEFAULT_EXCLUDE_DIRS for part in path.parts)


def scan_input_files(input_dir: Path, output_dir: Path, include_existing_cleaned: bool) -> list[FilePlan]:
    plans: list[FilePlan] = []
    for path in input_dir.rglob("*"):
        if not path.is_file():
            continue
        if path.suffix.lower() not in INPUT_SUFFIXES:
            continue
        if should_skip_path(path, output_dir, include_existing_cleaned):
            continue
        rel = path.relative_to(input_dir)
        rel_parent = "" if str(rel.parent) == "." else str(rel.parent)
        stem_slug = slugify(path.stem)
        plans.append(
            FilePlan(
                path=str(path),
                rel_parent=rel_parent,
                stem_slug=stem_slug,
                suffix=path.suffix.lower(),
                size_bytes=path.stat().st_size,
            )
        )
    return sorted(plans, key=lambda item: item.size_bytes, reverse=True)


def folder_size(path: Path) -> int:
    return sum(file.stat().st_size for file in path.rglob("*") if file.is_file())


def auto_worker_count(num_files: int) -> int:
    cpu_count = os.cpu_count() or 1
    physical_cores = psutil.cpu_count(logical=False) if psutil else None
    cores = physical_cores or max(1, cpu_count // 2)
    if psutil:
        ram_gb = psutil.virtual_memory().total / (1024**3)
        ram_limited = max(1, int(ram_gb // 4))
        cores = min(cores, ram_limited)
    return max(1, min(num_files, cores))


def log_hardware() -> None:
    cpu_count = os.cpu_count() or 1
    if psutil:
        physical = psutil.cpu_count(logical=False)
        ram_gb = psutil.virtual_memory().total / (1024**3)
        logging.info("Hardware detected: %s physical cores / %s threads, %.1f GB RAM", physical, cpu_count, ram_gb)
    else:
        logging.info("Hardware detected: %s CPU threads", cpu_count)


def write_run_summary(output_dir: Path, results: list[FileResult], args: argparse.Namespace) -> None:
    summary_path = output_dir / "_cleaning_summary.json"
    total_input_mb = sum(result.input_mb for result in results)
    total_output_mb = sum(result.output_mb for result in results)
    total_rows = sum(result.rows for result in results)
    payload = {
        "args": vars(args),
        "total_files": len(results),
        "successful_files": sum(result.error is None for result in results),
        "failed_files": sum(result.error is not None for result in results),
        "total_rows": total_rows,
        "total_input_mb": total_input_mb,
        "total_output_mb": total_output_mb,
        "compression_ratio_percent": (total_output_mb / total_input_mb * 100) if total_input_mb else 0,
        "results": [asdict(result) for result in results],
    }
    summary_path.write_text(json.dumps(payload, indent=2, ensure_ascii=True), encoding="utf-8")
    logging.info("Wrote summary: %s", summary_path)


def main() -> int:
    args = parse_args()
    configure_logging(args.log_level)

    input_dir = Path(args.input_folder).resolve()
    output_dir = Path(args.output_folder).resolve()
    if not input_dir.exists():
        logging.error("Input folder does not exist: %s", input_dir)
        return 2

    if args.overwrite and output_dir.exists():
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    log_hardware()
    plans = scan_input_files(input_dir, output_dir, args.include_existing_cleaned)
    if args.max_files > 0:
        plans = plans[: args.max_files]
    if not plans:
        logging.error("No supported files found under %s", input_dir)
        return 2

    total_gb = sum(plan.size_bytes for plan in plans) / 1_000_000_000
    workers = 1 if args.mode == "single-thread" else (args.num_workers or auto_worker_count(len(plans)))
    workers = max(1, min(workers, len(plans)))
    logging.info("Files: %s | input: %.2f GB | workers: %s | chunk rows: %s", len(plans), total_gb, workers, f"{args.chunk_size:,}")

    start = time.perf_counter()
    results: list[FileResult] = []
    args_dict = vars(args)

    if workers == 1:
        for idx, plan in enumerate(plans, start=1):
            logging.info("[%s/%s] Processing %s (%.2f MB)", idx, len(plans), plan.path, plan.size_bytes / 1_000_000)
            result = process_file(plan, str(output_dir), args_dict)
            results.append(result)
            log_result(result)
    else:
        with ProcessPoolExecutor(max_workers=workers) as pool:
            futures = {
                pool.submit(process_file, plan, str(output_dir), args_dict): plan
                for plan in plans
            }
            for idx, future in enumerate(as_completed(futures), start=1):
                result = future.result()
                results.append(result)
                logging.info("[%s/%s] Completed %s", idx, len(plans), Path(result.file).name)
                log_result(result)

    elapsed = time.perf_counter() - start
    write_run_summary(output_dir, results, args)

    failures = [result for result in results if result.error]
    total_rows = sum(result.rows for result in results)
    total_output_mb = sum(result.output_mb for result in results)
    logging.info(
        "Finished in %.1f min | rows: %s | output: %.2f GB | failures: %s",
        elapsed / 60,
        f"{total_rows:,}",
        total_output_mb / 1000,
        len(failures),
    )
    if failures:
        logging.error("Some files failed. See _cleaning_summary.json for details.")
        return 1
    return 0


def log_result(result: FileResult) -> None:
    if result.error:
        logging.error("%s failed: %s", result.file, result.error)
        return
    rate = result.input_mb / result.seconds if result.seconds > 0 else 0
    logging.info(
        "%s | rows=%s | parts=%s | output=%.2f MB | %.2f MB/s",
        Path(result.file).name,
        f"{result.rows:,}",
        result.parts,
        result.output_mb,
        rate,
    )




In [4]:
# ---- Run the cleaning pipeline ----------------------------------------------
args = argparse.Namespace(
    input_folder=str(INPUT_FOLDER),
    output_folder=str(OUTPUT_FOLDER),
    num_workers=NUM_WORKERS,
    chunk_size=CHUNK_SIZE,
    mode=MODE,
    engine=ENGINE,
    compression=COMPRESSION,
    overwrite=OVERWRITE,
    sample_rows=SAMPLE_ROWS,
    include_existing_cleaned=INCLUDE_EXISTING_CLEANED,
    max_files=MAX_FILES,
    log_level=LOG_LEVEL,
)

ensure_runtime_dependencies()
configure_logging(args.log_level)
input_dir = Path(args.input_folder).resolve()
output_dir = Path(args.output_folder).resolve()

if not input_dir.exists():
    raise FileNotFoundError(f'Input folder does not exist: {input_dir}')

if args.overwrite and output_dir.exists():
    shutil.rmtree(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

log_hardware()
plans = scan_input_files(input_dir, output_dir, args.include_existing_cleaned)
if args.max_files > 0:
    plans = plans[: args.max_files]
if not plans:
    raise RuntimeError(f'No supported files found under {input_dir}')

total_gb = sum(plan.size_bytes for plan in plans) / 1_000_000_000
workers = 1 if args.mode == 'single-thread' else (args.num_workers or auto_worker_count(len(plans)))
workers = max(1, min(workers, len(plans)))
logging.info('Files: %s | input: %.2f GB | workers: %s | chunk rows: %s', len(plans), total_gb, workers, f'{args.chunk_size:,}')

start = time.perf_counter()
results = []
args_dict = vars(args)

# NOTE: In Windows/Jupyter notebooks, ProcessPoolExecutor is often unstable
# because child processes cannot reliably import functions defined in notebook
# cells. The notebook therefore uses ThreadPoolExecutor for parallel file work.
# For true multiprocessing, run high_performance_dataset_cleaning_v3.py.
if workers == 1:
    for idx, plan in enumerate(plans, start=1):
        logging.info('[%s/%s] Processing %s (%.2f MB)', idx, len(plans), plan.path, plan.size_bytes / 1_000_000)
        result = process_file(plan, str(output_dir), args_dict)
        results.append(result)
        log_result(result)
else:
    logging.info('Notebook parallel mode uses ThreadPoolExecutor for Windows/Jupyter stability.')
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = {pool.submit(process_file, plan, str(output_dir), args_dict): plan for plan in plans}
        for idx, future in enumerate(as_completed(futures), start=1):
            result = future.result()
            results.append(result)
            logging.info('[%s/%s] Completed %s', idx, len(plans), Path(result.file).name)
            log_result(result)

elapsed = time.perf_counter() - start
write_run_summary(output_dir, results, args)

failures = [result for result in results if result.error]
total_rows = sum(result.rows for result in results)
total_output_mb = sum(result.output_mb for result in results)
logging.info(
    'Finished in %.1f min | rows: %s | output: %.2f GB | failures: %s',
    elapsed / 60,
    f'{total_rows:,}',
    total_output_mb / 1000,
    len(failures),
)

if failures:
    print('Failed files:')
    for failure in failures[:20]:
        print(f'- {failure.file}: {failure.error}')
    if len(failures) > 20:
        print(f'... and {len(failures) - 20} more. See _cleaning_summary.json')


2026-05-03 23:28:09 | INFO    | MainProcess | Hardware detected: 6 physical cores / 12 threads, 31.9 GB RAM
2026-05-03 23:28:10 | INFO    | MainProcess | Files: 1475 | input: 96.45 GB | workers: 4 | chunk rows: 500,000
2026-05-03 23:28:10 | INFO    | MainProcess | Notebook parallel mode uses ThreadPoolExecutor for Windows/Jupyter stability.
2026-05-03 23:28:10 | INFO    | MainProcess | Start processing E:\thesis\dataset\LANL\auth.txt\auth.txt
2026-05-03 23:28:10 | INFO    | MainProcess | Start processing E:\thesis\dataset\LANL\proc.txt\proc.txt
2026-05-03 23:28:10 | INFO    | MainProcess | Start processing E:\thesis\dataset\LANL\flows.txt\flows.txt
2026-05-03 23:28:10 | INFO    | MainProcess | Start processing E:\thesis\dataset\LANL\dns.txt\dns.txt
2026-05-03 23:28:11 | INFO    | MainProcess | dns.txt | wrote 1 chunks, 500,000 rows
2026-05-03 23:28:11 | INFO    | MainProcess | proc.txt | wrote 1 chunks, 500,000 rows
2026-05-03 23:28:13 | INFO    | MainProcess | flows.txt | wrote 1 chun

In [5]:
# ---- Load cleaned data examples ---------------------------------------------
# import pandas as pd
# df = pd.read_parquet(next(Path(OUTPUT_FOLDER).rglob('*.parquet')))
# display(df.head())

# Lazy scan all cleaned Parquet files with Polars:
# import polars as pl
# lf = pl.scan_parquet(str(Path(OUTPUT_FOLDER) / '**/*.parquet'))
# display(lf.head().collect())

summary_path = Path(OUTPUT_FOLDER) / '_cleaning_summary.json'
print('Summary:', summary_path if summary_path.exists() else 'not created yet')


Summary: E:\thesis\dataset\dataset_cleaned_parquet\_cleaning_summary.json
